# 1.导入相关的库

In [ ]:
import torch
import random
import optuna
import warnings
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from optuna.trial import TrialState
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 2.配置环境

In [ ]:
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)


def set_seed(seed=2025):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


RANDOM_SEED = 2025
set_seed(RANDOM_SEED)

# 3.加载数据文件

In [ ]:
train_df = pd.read_csv("ohio_t1dm_train_data.csv")
test_df = pd.read_csv("ohio_t1dm_test_data.csv")

feature_col = ["glucose", "insulin", "meal_carbs", "timestamp"]
patient_ids = [559, 563, 570, 575, 588, 591]

train_dfs = {}
val_dfs = {}
test_dfs = {}

for pid in patient_ids:
    train_patient = (
        train_df[train_df["patient_id"] == pid][feature_col]
        .sort_values("timestamp")
        .reset_index(drop=True)
    )
    train_part, val_part = train_test_split(train_patient, test_size=0.2, shuffle=False)
    train_dfs[pid] = train_part
    val_dfs[pid] = val_part.sort_values("timestamp").reset_index(drop=True)

    test_dfs[pid] = (
        test_df[test_df["patient_id"] == pid][feature_col]
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

# 4.滑动窗口构建数据集

In [ ]:
class SlidingWindowDataset(Dataset):
    def __init__(
        self,
        dataframe,
        window_minutes=60,
        forecast_minutes=30,
        data_interval_minutes=5,
        max_gap_minutes=10,
        scalers=None,
        fit_scalers=True,
        debug=False,
        verbose=True,
    ):
        self.df = dataframe.copy().sort_values("timestamp").reset_index(drop=True)
        self.window_minutes = window_minutes
        self.forecast_minutes = forecast_minutes
        self.data_interval = data_interval_minutes
        self.max_gap = max_gap_minutes
        self.fit_scalers = fit_scalers
        self.debug = debug
        self.verbose = verbose

        self.window_size = window_minutes // data_interval_minutes
        self.forecast_step = forecast_minutes // data_interval_minutes

        self._impute_missing_values()
        self._identify_gaps()
        self._extract_features()
        self._normalize_features(scalers)
        self.valid_indices = self._get_valid_indices()

        if self.verbose:
            self._validate_data_quality()

    def _impute_missing_values(self):
        self.df["timestamp"] = pd.to_datetime(self.df["timestamp"])
        if "glucose" not in self.df.columns:
            return

        is_na = self.df["glucose"].isna()
        if not is_na.any():
            if self.verbose:
                print("No missing values in glucose")
            return

        na_groups = (is_na != is_na.shift()).cumsum()
        self.df["_na_groups"] = na_groups

        filled_count = 0
        kept_count = 0

        for group_id in self.df[is_na]["_na_groups"].unique():
            group_mask = (self.df["_na_groups"] == group_id) & is_na
            group_indices = self.df[group_mask].index.tolist()

            if not group_indices:
                continue

            gap_duration = len(group_indices) * self.data_interval

            if gap_duration <= self.max_gap:
                self._fill_gap("glucose", group_indices, is_na)
                filled_count += len(group_indices)
            else:
                kept_count += len(group_indices)

        self.df.drop("_na_groups", axis=1, inplace=True, errors="ignore")

        if self.verbose:
            print(
                f"Glucose imputation: filled {filled_count} NaN, kept {kept_count} NaN for filtering"
            )

    def _fill_gap(self, feature, indices, is_na):
        start_idx = indices[0]
        end_idx = indices[-1]

        prev_valid_idx = start_idx - 1
        while prev_valid_idx >= 0 and is_na.iloc[prev_valid_idx]:
            prev_valid_idx -= 1

        next_valid_idx = end_idx + 1
        while next_valid_idx < len(self.df) and is_na.iloc[next_valid_idx]:
            next_valid_idx += 1

        if prev_valid_idx >= 0 and next_valid_idx < len(self.df):
            prev_val = self.df.iloc[prev_valid_idx][feature]
            next_val = self.df.iloc[next_valid_idx][feature]

            for idx in range(start_idx, end_idx + 1):
                ratio = (idx - prev_valid_idx) / (next_valid_idx - prev_valid_idx)
                self.df.at[self.df.index[idx], feature] = (
                    prev_val + (next_val - prev_val) * ratio
                )

        elif prev_valid_idx >= 0:
            self._extrapolate_forward(
                feature, start_idx, end_idx, prev_valid_idx, is_na
            )
        elif next_valid_idx < len(self.df):
            self._extrapolate_backward(
                feature, start_idx, end_idx, next_valid_idx, is_na
            )

    def _extrapolate_forward(self, feature, start_idx, end_idx, last_valid_idx, is_na):
        second_last_idx = last_valid_idx - 1
        while second_last_idx >= 0 and is_na.iloc[second_last_idx]:
            second_last_idx -= 1

        if second_last_idx >= 0:
            val2 = self.df.iloc[second_last_idx][feature]
            val1 = self.df.iloc[last_valid_idx][feature]
            slope = val1 - val2

            for idx in range(start_idx, end_idx + 1):
                steps = idx - last_valid_idx
                self.df.at[self.df.index[idx], feature] = val1 + slope * steps
        else:
            fill_val = self.df.iloc[last_valid_idx][feature]
            self.df.loc[self.df.index[start_idx : end_idx + 1], feature] = fill_val

    def _extrapolate_backward(
        self, feature, start_idx, end_idx, first_valid_idx, is_na
    ):
        second_valid_idx = first_valid_idx + 1
        while second_valid_idx < len(self.df) and is_na.iloc[second_valid_idx]:
            second_valid_idx += 1

        if second_valid_idx < len(self.df):
            val1 = self.df.iloc[first_valid_idx][feature]
            val2 = self.df.iloc[second_valid_idx][feature]
            slope = val2 - val1

            for idx in range(start_idx, end_idx + 1):
                steps = idx - first_valid_idx
                self.df.at[self.df.index[idx], feature] = val1 + slope * steps
        else:
            fill_val = self.df.iloc[first_valid_idx][feature]
            self.df.loc[self.df.index[start_idx : end_idx + 1], feature] = fill_val

    def _identify_gaps(self):
        self.df["timestamp"] = pd.to_datetime(self.df["timestamp"])
        self.df["time_diff"] = self.df["timestamp"].diff().dt.total_seconds() / 60
        self.df["is_gap"] = self.df["time_diff"] > self.max_gap
        self.df["gap_id"] = self.df["is_gap"].cumsum()

    def _extract_features(self):
        self.df["hour"] = self.df["timestamp"].dt.hour
        self.df["minute"] = self.df["timestamp"].dt.minute
        self.df["day_of_week"] = self.df["timestamp"].dt.dayofweek
        self.df["is_weekend"] = self.df["day_of_week"].isin([5, 6]).astype(float)

        self.df["hour_sin"] = np.sin(2 * np.pi * self.df["hour"] / 24)
        self.df["hour_cos"] = np.cos(2 * np.pi * self.df["hour"] / 24)

        self.numeric_features = ["glucose", "insulin", "meal_carbs"]
        self.time_features = ["hour_sin", "hour_cos", "is_weekend"]
        self.all_features = self.numeric_features + self.time_features

    def _get_valid_indices(self):
        valid_indices = []
        self.drop_stats = {
            "segment_too_short": 0,
            "window_has_nan": 0,
            "target_is_nan": 0,
        }

        for _, group_data in self.df.groupby("gap_id"):
            group_indices = group_data.index.tolist()

            if len(group_indices) < self.window_size + self.forecast_step:
                self.drop_stats["segment_too_short"] += len(group_indices)
                continue

            max_start_idx = len(group_indices) - self.window_size - self.forecast_step

            for i in range(max_start_idx + 1):
                start_idx = group_indices[i]
                window_end_idx = start_idx + self.window_size - 1
                target_idx = start_idx + self.window_size + self.forecast_step - 1

                window_glucose = self.df.loc[start_idx:window_end_idx, "glucose"]
                if window_glucose.isna().any():
                    self.drop_stats["window_has_nan"] += 1
                    continue

                target_value = self.df.loc[target_idx, "glucose"]
                if pd.isna(target_value):
                    self.drop_stats["target_is_nan"] += 1
                    continue

                valid_indices.append(start_idx)

        return valid_indices

    def _normalize_features(self, external_scalers=None):
        self.scalers = {}

        for feature in self.numeric_features:
            if self.fit_scalers:
                scaler = RobustScaler()
                non_na_mask = self.df[feature].notna()

                if non_na_mask.any():
                    self.df.loc[non_na_mask, feature] = scaler.fit_transform(
                        self.df.loc[non_na_mask, feature].values.reshape(-1, 1)
                    ).flatten()
                self.scalers[feature] = scaler
            else:
                if external_scalers is None or feature not in external_scalers:
                    raise ValueError(f"Missing scaler for: {feature}")

                scaler = external_scalers[feature]
                non_na_mask = self.df[feature].notna()

                if non_na_mask.any():
                    self.df.loc[non_na_mask, feature] = scaler.transform(
                        self.df.loc[non_na_mask, feature].values.reshape(-1, 1)
                    ).flatten()
                self.scalers[feature] = scaler

        self.feature_data = torch.tensor(
            self.df[self.all_features].values, dtype=torch.float32
        )
        self.target_data = torch.tensor(self.df["glucose"].values, dtype=torch.float32)

    def _validate_data_quality(self):
        print(f"\n{'=' * 60}")
        print("Data Quality Report")
        print(f"{'=' * 60}")

        print("\n[Original Missing Values]")
        for feature in ["glucose", "insulin", "meal_carbs"]:
            if feature in self.df.columns:
                nan_count = self.df[feature].isna().sum()
                nan_pct = 100 * nan_count / len(self.df) if len(self.df) > 0 else 0
                print(f"  {feature:12s}: {nan_count:5d} ({nan_pct:5.2f}%)")

        print("\n[Sample Generation]")
        print(f"  Total rows:       {len(self.df):5d}")
        print(f"  Valid samples:    {len(self.valid_indices):5d}")

        if len(self.df) > 0:
            drop_rate = 100 * (1 - len(self.valid_indices) / len(self.df))
            print(f"  Drop rate:        {drop_rate:5.2f}%")

        if hasattr(self, "drop_stats"):
            print("\n[Drop Reasons]")
            for reason, count in self.drop_stats.items():
                print(f"  {reason:20s}: {count:5d}")

        print(f"{'=' * 60}\n")

    def get_scalers(self):
        return self.scalers

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        x = self.feature_data[actual_idx : actual_idx + self.window_size]
        y = self.target_data[actual_idx + self.window_size + self.forecast_step - 1]

        if self.debug and (torch.isnan(x).any() or torch.isnan(y)):
            raise ValueError(f"NaN detected in sample {idx}")

        return x, y

# 5.构建序列模型

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, output_size),
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_output = lstm_out[:, -1, :]
        output = self.fc(last_output)
        return output

In [ ]:
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout=0.2):
        super(GRUModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, output_size),
        )

    def forward(self, x):
        gru_out, _ = self.gru(x)
        last_output = gru_out[:, -1, :]
        output = self.fc(last_output)
        return output

In [ ]:
class TransformerModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout=0.2):
        super(TransformerModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.input_size = input_size
        effective_hidden = hidden_size * 2
        self.input_projection = nn.Linear(input_size, effective_hidden)
        self.positional_encoding = nn.Parameter(
            torch.zeros(1, 5000, effective_hidden), requires_grad=True
        )
        nn.init.normal_(self.positional_encoding, mean=0, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=effective_hidden,
            nhead=8,
            dim_feedforward=effective_hidden * 2,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        effective_num_layers = max(2, num_layers)
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=effective_num_layers,
            norm=nn.LayerNorm(effective_hidden),
        )

        self.fc = nn.Sequential(
            nn.Linear(effective_hidden, effective_hidden // 2),
            nn.LayerNorm(effective_hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(effective_hidden // 2, output_size),
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        _, seq_len, _ = x.shape
        x = self.input_projection(x)
        x = x + self.positional_encoding[:, :seq_len, :]
        x = self.dropout(x)
        transformer_out = self.transformer_encoder(x)
        last_output = transformer_out[:, -1, :]
        output = self.fc(last_output)

        return output

# 6.训练验证调参测试

In [ ]:
class MultiPatientPredictor:
    def __init__(self, train_dfs_dict, val_dfs_dict):
        self.train_dfs_dict = {k: v.copy() for k, v in train_dfs_dict.items()}
        self.val_dfs_dict = {k: v.copy() for k, v in val_dfs_dict.items()}
        self.patient_ids = list(train_dfs_dict.keys())
        self.model = None
        self.global_scalers = None

        self.train_datasets = {}
        self.val_datasets = {}

    def prepare_data(
        self,
        window_minutes=60,
        forecast_minutes=30,
        data_interval_minutes=5,
        max_gap_minutes=10,
    ):
        print("\n" + "=" * 80)
        print("Preparing multi-patient data...")
        print("=" * 80)

        print("\n[Step 1] Fitting global scaler on combined training data...")
        combined_train_df = pd.concat(
            list(self.train_dfs_dict.values()), ignore_index=True
        )

        temp_dataset = SlidingWindowDataset(
            dataframe=combined_train_df,
            window_minutes=window_minutes,
            forecast_minutes=forecast_minutes,
            data_interval_minutes=data_interval_minutes,
            max_gap_minutes=max_gap_minutes,
            scalers=None,
            fit_scalers=True,
            verbose=False,
        )
        self.global_scalers = temp_dataset.get_scalers()
        print("Global scaler fitting complete!")

        print("\n[Step 2] Building training dataset for each patient...")
        for pid in self.patient_ids:
            print(f"  Processing patient {pid}...", end=" ")
            self.train_datasets[pid] = SlidingWindowDataset(
                dataframe=self.train_dfs_dict[pid],
                window_minutes=window_minutes,
                forecast_minutes=forecast_minutes,
                data_interval_minutes=data_interval_minutes,
                max_gap_minutes=max_gap_minutes,
                scalers=self.global_scalers,
                fit_scalers=False,
                verbose=False,
            )
            print(f"Done. ({len(self.train_datasets[pid])} samples)")

        print("\n[Step 3] Building validation dataset for each patient...")
        for pid in self.patient_ids:
            print(f"  Processing patient {pid}...", end=" ")
            self.val_datasets[pid] = SlidingWindowDataset(
                dataframe=self.val_dfs_dict[pid],
                window_minutes=window_minutes,
                forecast_minutes=forecast_minutes,
                data_interval_minutes=data_interval_minutes,
                max_gap_minutes=max_gap_minutes,
                scalers=self.global_scalers,
                fit_scalers=False,
                verbose=False,
            )
            print(f"Done. ({len(self.val_datasets[pid])} samples)")

        print("\n" + "=" * 80)
        total_train = sum(len(ds) for ds in self.train_datasets.values())
        total_val = sum(len(ds) for ds in self.val_datasets.values())
        print(f"Total training samples: {total_train}")
        print(f"Total validation samples: {total_val}")
        print("=" * 80)

    def create_dataloaders(self, batch_size=64, num_workers=0):
        combined_train = ConcatDataset(list(self.train_datasets.values()))
        combined_val = ConcatDataset(list(self.val_datasets.values()))

        self.train_loader = DataLoader(
            combined_train,
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers,
            pin_memory=False,
        )

        self.val_loader = DataLoader(
            combined_val,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            pin_memory=False,
        )

        return self.train_loader, self.val_loader

    def optimize_hyperparameters(
        self,
        model_type,
        n_trials=50,
        device="cuda" if torch.cuda.is_available() else "cpu",
    ):
        print(f"\n{'=' * 80}")
        print(f"Starting hyperparameter optimization for {model_type} model")
        print(f"{'=' * 80}")

        first_pid = self.patient_ids[0]
        input_size = self.train_datasets[first_pid].feature_data.shape[1]

        def objective(trial):
            hidden_size = trial.suggest_categorical("hidden_size", [16, 32, 64, 128, 256])
            num_layers = trial.suggest_int("num_layers", 1, 6)
            dropout = trial.suggest_float("dropout", 0.1, 0.5)
            learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-1)
            batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128, 256])
            weight_decay = trial.suggest_loguniform("weight_decay", 1e-6, 1e-1)

            combined_train = ConcatDataset(list(self.train_datasets.values()))
            combined_val = ConcatDataset(list(self.val_datasets.values()))

            train_loader = DataLoader(
                combined_train,
                batch_size=batch_size,
                shuffle=True,
                num_workers=0,
                pin_memory=False,
            )

            val_loader = DataLoader(
                combined_val,
                batch_size=batch_size,
                shuffle=False,
                num_workers=0,
                pin_memory=False,
            )

            if model_type == "LSTM":
                model = LSTMModel(
                    input_size=input_size,
                    hidden_size=hidden_size,
                    num_layers=num_layers,
                    output_size=1,
                    dropout=dropout,
                ).to(device)
            elif model_type == "GRU":
                model = GRUModel(
                    input_size=input_size,
                    hidden_size=hidden_size,
                    num_layers=num_layers,
                    output_size=1,
                    dropout=dropout,
                ).to(device)
            elif model_type == "Transformer":
                model = TransformerModel(
                    input_size=input_size,
                    hidden_size=hidden_size,
                    num_layers=num_layers,
                    output_size=1,
                    dropout=dropout,
                ).to(device)

            criterion = nn.MSELoss()
            optimizer = optim.AdamW(
                model.parameters(), lr=learning_rate, weight_decay=weight_decay
            )

            best_val_loss = float("inf")
            patience = 25
            patience_counter = 0
            max_epochs = 100

            for epoch in range(max_epochs):
                model.train()
                for x_batch, y_batch in train_loader:
                    x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                    optimizer.zero_grad()
                    predictions = model(x_batch).squeeze()
                    loss = criterion(predictions, y_batch)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()

                model.eval()
                val_loss = 0.0
                val_batches = 0

                with torch.no_grad():
                    for x_batch, y_batch in val_loader:
                        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                        predictions = model(x_batch).squeeze()
                        loss = criterion(predictions, y_batch)
                        val_loss += loss.item()
                        val_batches += 1

                avg_val_loss = val_loss / val_batches

                if avg_val_loss < best_val_loss:
                    best_val_loss = avg_val_loss
                    patience_counter = 0
                else:
                    patience_counter += 1

                if patience_counter >= patience:
                    break

                trial.report(avg_val_loss, epoch)
                if trial.should_prune():
                    raise optuna.TrialPruned()

            return best_val_loss

        study = optuna.create_study(
            direction="minimize",
            pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10),
        )

        study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

        print("\nBest hyperparameters:")
        for key, value in study.best_params.items():
            print(f"  {key}: {value}")
        print(f"Best validation loss: {study.best_value * 1e4:.4f}")

        return study.best_params

    def train_model(
        self,
        model_type="LSTM",
        hidden_size=64,
        num_layers=2,
        dropout=0.2,
        learning_rate=1e-3,
        batch_size=64,
        weight_decay=1e-4,
        epochs=100,
        patience=25,
        device="cuda" if torch.cuda.is_available() else "cpu",
    ):
        if self.train_loader is None:
            self.create_dataloaders(batch_size=batch_size)

        print("\n" + "=" * 80)
        print(f"Starting training of {model_type} model...")
        print("=" * 80)

        first_pid = self.patient_ids[0]
        input_size = self.train_datasets[first_pid].feature_data.shape[1]

        if model_type == "LSTM":
            self.model = LSTMModel(
                input_size=input_size,
                hidden_size=hidden_size,
                num_layers=num_layers,
                output_size=1,
                dropout=dropout,
            ).to(device)
        elif model_type == "GRU":
            self.model = GRUModel(
                input_size=input_size,
                hidden_size=hidden_size,
                num_layers=num_layers,
                output_size=1,
                dropout=dropout,
            ).to(device)
        elif model_type == "Transformer":
            self.model = TransformerModel(
                input_size=input_size,
                hidden_size=hidden_size,
                num_layers=num_layers,
                output_size=1,
                dropout=dropout,
            ).to(device)

        criterion = nn.MSELoss()
        optimizer = optim.AdamW(
            self.model.parameters(), lr=learning_rate, weight_decay=weight_decay
        )
        scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=10, T_mult=2, eta_min=1e-8
        )

        best_val_loss = float("inf")
        best_model_state = None
        patience_counter = 0
        train_losses = []
        val_losses = []

        for epoch in range(epochs):
            self.model.train()
            train_loss = 0.0
            train_batches = 0

            for x_batch, y_batch in self.train_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)

                optimizer.zero_grad()
                predictions = self.model(x_batch).squeeze()
                loss = criterion(predictions, y_batch)
                loss.backward()

                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                optimizer.step()

                train_loss += loss.item()
                train_batches += 1

            avg_train_loss = train_loss / train_batches

            self.model.eval()
            val_loss = 0.0
            val_batches = 0

            with torch.no_grad():
                for x_batch, y_batch in self.val_loader:
                    x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                    predictions = self.model(x_batch).squeeze()
                    loss = criterion(predictions, y_batch)

                    val_loss += loss.item()
                    val_batches += 1

            avg_val_loss = val_loss / val_batches

            train_losses.append(avg_train_loss)
            val_losses.append(avg_val_loss)

            scheduler.step()

            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                best_model_state = self.model.state_dict().copy()
                patience_counter = 0
            else:
                patience_counter += 1

            if (epoch + 1) % 10 == 0 or epoch == 0:
                print(
                    f"Epoch [{epoch + 1:3d}/{epochs}] - "
                    f"Train loss: {avg_train_loss * 1e4:.4f}, "
                    f"Val loss: {avg_val_loss * 1e4:.4f}, "
                    f"Best val loss: {best_val_loss * 1e4:.4f}"
                )

            if patience_counter >= patience:
                print(f"\n⏹ Early stopping at epoch {epoch + 1}")
                break

        self.model.load_state_dict(best_model_state)
        print(f"\nTraining complete! Best validation loss: {best_val_loss * 1e4:.6f}")
        print("=" * 80)

        return train_losses, val_losses

    def test_on_all_patients(
        self,
        test_dfs_dict,
        device="cuda" if torch.cuda.is_available() else "cpu",
        batch_size=64,
    ):
        if self.model is None:
            raise ValueError("Model has not been trained yet")

        print("\n" + "=" * 80)
        print("Testing on all patients' test sets")
        print("=" * 80)

        results = {}
        all_metrics = {"mae": [], "rmse": [], "mape": [], "r2": []}

        glucose_scaler = self.global_scalers["glucose"]

        first_pid = self.patient_ids[0]
        dataset_config = {
            "window_minutes": self.train_datasets[first_pid].window_minutes,
            "forecast_minutes": self.train_datasets[first_pid].forecast_minutes,
            "data_interval_minutes": self.train_datasets[first_pid].data_interval,
            "max_gap_minutes": self.train_datasets[first_pid].max_gap,
        }

        for pid in self.patient_ids:
            print(f"\n{'─' * 60}")
            print(f"Testing patient: {pid}")

            test_dataset = SlidingWindowDataset(
                dataframe=test_dfs_dict[pid],
                **dataset_config,
                scalers=self.global_scalers,
                fit_scalers=False,
                verbose=False,
            )

            if len(test_dataset) == 0:
                print("No valid samples, skipping!")
                continue

            test_loader = DataLoader(
                test_dataset, batch_size=batch_size, shuffle=False, num_workers=0
            )

            self.model.eval()
            patient_predictions = []
            patient_targets = []

            with torch.no_grad():
                for x_batch, y_batch in test_loader:
                    x_batch = x_batch.to(device)
                    predictions = self.model(x_batch).squeeze()
                    patient_predictions.extend(predictions.cpu().numpy())
                    patient_targets.extend(y_batch.numpy())

            patient_predictions = np.array(patient_predictions).reshape(-1, 1)
            patient_targets = np.array(patient_targets).reshape(-1, 1)

            predictions_orig = glucose_scaler.inverse_transform(
                patient_predictions
            ).flatten()
            targets_orig = glucose_scaler.inverse_transform(patient_targets).flatten()

            mae = mean_absolute_error(targets_orig, predictions_orig)
            rmse = np.sqrt(mean_squared_error(targets_orig, predictions_orig))
            mape = (
                np.mean(np.abs((targets_orig - predictions_orig) / targets_orig)) * 100
            )
            r2 = r2_score(targets_orig, predictions_orig)

            results[pid] = {
                "mae": mae,
                "rmse": rmse,
                "mape": mape,
                "r2": r2,
                "predictions": predictions_orig,
                "targets": targets_orig,
                "n_samples": len(targets_orig),
            }

            all_metrics["mae"].append(mae)
            all_metrics["rmse"].append(rmse)
            all_metrics["mape"].append(mape)
            all_metrics["r2"].append(r2)

            print(f"  Number of samples: {len(targets_orig)}")
            print(f"  MAE:    {mae:7.2f} mg/dL")
            print(f"  RMSE:   {rmse:7.2f} mg/dL")
            print(f"  MAPE:   {mape:7.2f}%")
            print(f"  R²:     {r2:7.4f}")

        print("\n" + "=" * 80)
        print("Overall statistics across all patients")
        print("=" * 80)

        for metric_name, display in [
            ("mae", "MAE (mg/dL)"),
            ("rmse", "RMSE (mg/dL)"),
            ("mape", "MAPE (%)"),
            ("r2", "R²"),
        ]:
            values = all_metrics[metric_name]
            print(f"\n{display}:")
            print(f"  Mean ± SD: {np.mean(values):7.2f} ± {np.std(values):7.2f}")
            print(f"  Range:    [{np.min(values):7.2f}, {np.max(values):7.2f}]")

        print("\n" + "=" * 80)

        results["overall"] = {
            "mae_mean": np.mean(all_metrics["mae"]),
            "mae_std": np.std(all_metrics["mae"]),
            "rmse_mean": np.mean(all_metrics["rmse"]),
            "rmse_std": np.std(all_metrics["rmse"]),
            "mape_mean": np.mean(all_metrics["mape"]),
            "mape_std": np.std(all_metrics["mape"]),
            "r2_mean": np.mean(all_metrics["r2"]),
            "r2_std": np.std(all_metrics["r2"]),
            "n_patients": len(all_metrics["mae"]),
        }

        return results

    def save_model(self, filepath):
        if self.model is None:
            raise ValueError("No model to save")

        first_pid = self.patient_ids[0]

        save_dict = {
            "model_state_dict": self.model.state_dict(),
            "model_config": {
                "input_size": self.train_datasets[first_pid].feature_data.shape[1],
                "hidden_size": self.model.hidden_size,
                "num_layers": self.model.num_layers,
            },
            "scalers": self.global_scalers,
            "dataset_config": {
                "window_minutes": self.train_datasets[first_pid].window_minutes,
                "forecast_minutes": self.train_datasets[first_pid].forecast_minutes,
                "data_interval": self.train_datasets[first_pid].data_interval,
                "max_gap": self.train_datasets[first_pid].max_gap,
            },
            "patient_ids": self.patient_ids,
        }

        torch.save(save_dict, filepath)
        print(f"Model saved to: {filepath}")

# 7.运行与展示

In [ ]:
print("\n" + "=" * 80)
print("Multi-patient Glucose Prediction Model Training and Evaluation")
print("Comparison of Three Models")
print("=" * 80)

In [ ]:
all_model_results = {}
model_types = ["LSTM", "GRU", "Transformer"]

In [ ]:
for model_type in model_types:
    print(f"{'#' * 80}")
    print(f"Processing Model: {model_type}")
    print(f"{'#' * 80}")

    predictor = MultiPatientPredictor(train_dfs_dict=train_dfs, val_dfs_dict=val_dfs)

    predictor.prepare_data(
        window_minutes=60,
        forecast_minutes=30,
        data_interval_minutes=5,
        max_gap_minutes=10,
    )

    # Step 1: Optuna hyperparameter optimization
    best_params = predictor.optimize_hyperparameters(
        model_type=model_type,
        n_trials=50,
        device="cuda" if torch.cuda.is_available() else "cpu",
    )

    predictor.create_dataloaders(batch_size=best_params["batch_size"])

    # Step 2: Train model with best hyperparameters
    train_losses, val_losses = predictor.train_model(
        model_type=model_type,
        hidden_size=best_params["hidden_size"],
        num_layers=best_params["num_layers"],
        dropout=best_params["dropout"],
        learning_rate=best_params["learning_rate"],
        batch_size=best_params["batch_size"],
        weight_decay=best_params["weight_decay"],
        epochs=100,
        patience=30,
        device="cuda" if torch.cuda.is_available() else "cpu",
    )

    # Step 3: Test set evaluation
    test_results = predictor.test_on_all_patients(test_dfs)

    # Step 4: Save model
    predictor.save_model(f"save_model/{model_type}_model_30_optimized.pth")

    # Save results
    all_model_results[model_type] = {
        "best_params": best_params,
        "test_results": test_results,
    }

    # Print detailed results for this model
    print(f"{'=' * 80}")
    print(f"{model_type} Model - Detailed Test Results")
    print(f"{'=' * 80}\n")

    print("[Best Hyperparameters]")
    for param, value in best_params.items():
        print(f"  {param}: {value}")

    print("\n[Test Results by Patient]")
    print(
        f"{'Patient ID':^15} {'N Samples':^15} {'MAE':^15} {'RMSE':^15} {'MAPE':^15} {'R²':^15}"
    )
    print("─" * 100)

    for pid in patient_ids:
        if pid in test_results:
            r = test_results[pid]
            print(
                f"{pid:^14} {r['n_samples']:^18} {r['mae']:^16.2f} {r['rmse']:^13.2f} {r['mape']:^16.2f} {r['r2']:^14.2f}"
            )

    print("─" * 100)

    overall = test_results["overall"]
    print("\n[Overall Statistics]")
    print(f"  MAE:  {overall['mae_mean']:.2f} ± {overall['mae_std']:.2f} mg/dL")
    print(f"  RMSE: {overall['rmse_mean']:.2f} ± {overall['rmse_std']:.2f} mg/dL")
    print(f"  MAPE: {overall['mape_mean']:.2f} ± {overall['mape_std']:.2f}%")
    print(f"  R²:   {overall['r2_mean']:.4f} ± {overall['r2_std']:.4f}")
    print(f"  Number of patients: {overall['n_patients']}")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    del predictor

In [ ]:
print("=" * 80)
print("Final Summary: Performance Comparison of Three Models")
print("=" * 80)

print(
    f"\n{'Model':^15} {'MAE (mg/dL)':^20} {'RMSE (mg/dL)':^20} {'MAPE (%)':^20} {'R²':^20}"
)
print("─" * 100)

for model_type in model_types:
    overall = all_model_results[model_type]["test_results"]["overall"]
    print(
        f"{model_type:^15} "
        f"{overall['mae_mean']:7.2f}±{overall['mae_std']:5.2f}     "
        f"{overall['rmse_mean']:7.2f}±{overall['rmse_std']:5.2f}     "
        f"{overall['mape_mean']:7.2f}±{overall['mape_std']:5.2f}     "
        f"{overall['r2_mean']:7.4f}±{overall['r2_std']:6.4f}"
    )

print("─" * 100)

best_model = min(
    model_types,
    key=lambda m: all_model_results[m]["test_results"]["overall"]["mae_mean"],
)

print(f"\n🏆 Best Model (based on MAE): {best_model}")
print(
    f"   MAE: {all_model_results[best_model]['test_results']['overall']['mae_mean']:.2f} mg/dL"
)

print("\n" + "=" * 80)
print("All tasks completed successfully!")
print("=" * 80)